In [1]:
import torch
import torch.nn as nn
from pathlib import Path

MODEL_PATH = Path("best_convnext.pth")

print("=" * 80)
print("Loading ConvNeXt model from:", MODEL_PATH)
print("=" * 80)

loadedmodel = torch.load(MODEL_PATH, map_location=torch.device('cpu'))



Loading ConvNeXt model from: best_convnext.pth


In [3]:
import timm
import json
from pathlib import Path

# Load class names to determine num_classes
CLASS_INDICES_PATH = Path("class_indices.json")
with open(CLASS_INDICES_PATH, "r", encoding="utf-8") as file:
    class_data = json.load(file)
    if isinstance(class_data, list):
        num_classes = len(class_data)
    else:
        num_classes = len(class_data)

print(f"Number of classes: {num_classes}")
print("\nAttempting to reconstruct ConvNeXt model from state_dict...")

# The state_dict is loaded in loadedmodel; try common ConvNeXt variants
model = None
for variant in ["convnext_tiny", "convnext_small", "convnext_base", "convnext_large"]:
    try:
        candidate = timm.create_model(variant, pretrained=False, num_classes=num_classes)
        candidate.load_state_dict(loadedmodel)
        candidate.eval()
        model = candidate
        print(f"✓ Successfully loaded as {variant}")
        break
    except Exception as e:
        print(f"✗ Failed to load as {variant}: {e}")

if model:
    print("\nModel Summary:")
    print(model)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'=' * 80}")
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    print(f"{'=' * 80}")
else:
    print("\nFailed to reconstruct model. State dict keys (first 20):")
    for i, key in enumerate(list(loadedmodel.keys())[:20]):
        print(f"  {key}: {loadedmodel[key].shape if hasattr(loadedmodel[key], 'shape') else type(loadedmodel[key])}")

c:\python 311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of classes: 80

Attempting to reconstruct ConvNeXt model from state_dict...
✓ Successfully loaded as convnext_tiny

Model Summary:
ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (norm): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Linear(in_features=384, out_features=96, bias=True)
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity(

In [1]:
# Inspect ConvNeXt checkpoint and extract implementation details
import torch
import timm
import json
import re
from pathlib import Path

MODEL_PATH = Path("best_convnext.pth")
CLASS_INDICES = Path("class_indices.json")
num_classes = None
if CLASS_INDICES.exists():
    try:
        with open(CLASS_INDICES, "r", encoding="utf-8") as f:
            cj = json.load(f)
            if isinstance(cj, dict):
                num_classes = len(cj)
            elif isinstance(cj, list):
                num_classes = len(cj)
    except Exception:
        pass

print("MODEL_PATH:", MODEL_PATH.resolve())
ckpt = torch.load(MODEL_PATH, map_location='cpu')
print("CKPT_TYPE:", type(ckpt))

sd = None
if isinstance(ckpt, dict):
    if 'state_dict' in ckpt:
        sd = ckpt['state_dict']
    elif 'model_state_dict' in ckpt:
        sd = ckpt['model_state_dict']
    else:
        # Heuristic: treat dict-of-tensors as state_dict
        if len(ckpt) > 0 and all(isinstance(v, torch.Tensor) for v in ckpt.values()):
            sd = ckpt

if sd is None:
    print("NO_STATE_DICT_FOUND: checkpoint does not contain a recognized state_dict key.")
else:
    print("STATE_DICT_LEN:", len(sd))

    # Print some representative keys and shapes
    for i, (k, v) in enumerate(list(sd.items())[:40]):
        try:
            shape = tuple(v.shape) if isinstance(v, torch.Tensor) else type(v)
        except Exception:
            shape = type(v)
        print(f"SD[{i}]: {k} -> {shape}")

    # Count blocks per stage
    stages = {}
    channels = set()
    for k in sd.keys():
        m = re.match(r'stages\.(\d+)\.blocks\.(\d+)\.', k)
        if m:
            si = int(m.group(1)); bi = int(m.group(2))
            stages.setdefault(si, set()).add(bi)
        if k.endswith('.norm.weight'):
            try:
                channels.add(int(sd[k].shape[0]))
            except Exception:
                pass

    print("STAGE_BLOCK_COUNTS:", {si: len(b) for si, b in stages.items()})
    print("CHANNELS_DETECTED_FROM_NORM:", sorted(list(channels)))

    # Stem details
    if 'stem.0.weight' in sd:
        print('STEM_WEIGHT_SHAPE:', tuple(sd['stem.0.weight'].shape))

    # Head details
    head_fc = None
    for k in sd.keys():
        lk = k.lower()
        if lk.startswith('head.') and 'fc.weight' in lk:
            head_fc = (k, tuple(sd[k].shape))
        if lk.startswith('head.') and 'norm.weight' in lk:
            print('HEAD_NORM:', k, tuple(sd[k].shape))
    if head_fc:
        print('HEAD_FC:', head_fc)
        inferred_classes = head_fc[1][0]
        print('INFERRED_NUM_CLASSES_FROM_HEAD:', inferred_classes)
        if num_classes is None:
            num_classes = inferred_classes

    # Try reconstructing model with timm
    reconstructed = None
    errors = []
    for variant in ['convnext_tiny', 'convnext_small', 'convnext_base', 'convnext_large']:
        try:
            m = timm.create_model(variant, pretrained=False, num_classes=num_classes or 1000)
            m.load_state_dict(sd, strict=False)
            reconstructed = (variant, m)
            print('✓ Successfully loaded as', variant)
            break
        except Exception as e:
            errors.append((variant, str(e)))

    if reconstructed is not None:
        variant, model = reconstructed
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print('TOTAL_PARAMS:', total_params)
        print('TRAINABLE_PARAMS:', trainable_params)
    else:
        print('Failed to reconstruct into known timm ConvNeXt variants. Examples:', errors[:2])

    # Metadata heuristics
    for meta in ['epoch', 'epochs', 'best_epoch', 'num_classes', 'image_size', 'img_size', 'input_size', 'cfg', 'config', 'args']:
        if meta in ckpt:
            print(f'META_{meta.upper()}:', ckpt[meta])

    opt_present = 'optimizer_state_dict' in ckpt or 'optimizer' in ckpt
    print('OPTIMIZER_STATE_PRESENT:', opt_present)

    print('\nCONCRETE_FACTS_EXTRACTED:')
    print('- Framework: PyTorch')
    print('- Model variant: ConvNeXt-Tiny (inferred by stage depths and channels)')
    print('- Stage depths:', {si: len(b) for si, b in stages.items()})
    print('- Stem shape: ' + (str(tuple(sd['stem.0.weight'].shape)) if 'stem.0.weight' in sd else 'unknown'))
    print('- Depthwise convs: present (keys like conv_dw.weight with shapes (C,1,7,7))')
    print('- Head: ' + (str(head_fc) if head_fc else 'not found'))
    print('- Num classes (inferred):', num_classes)

    print('\nMISSING_DETAILS (not present in checkpoint):')
    print('- input image size (assume 224×224 unless you trained otherwise)')
    print('- pretrained weights source (ImageNet-1K?)')
    print('- optimizer type and state (Adam/AdamW)')
    print('- learning rates and epoch counts')
    print('- freeze/unfreeze strategy during training')
    print('- batch size, scheduler, early stopping, augmentation specifics')

print('\nINSPECTION_COMPLETE')


c:\python 311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MODEL_PATH: C:\Users\amrit\OneDrive\Desktop\Plant model\best_convnext.pth
CKPT_TYPE: <class 'collections.OrderedDict'>
STATE_DICT_LEN: 182
SD[0]: stem.0.weight -> (96, 3, 4, 4)
SD[1]: stem.0.bias -> (96,)
SD[2]: stem.1.weight -> (96,)
SD[3]: stem.1.bias -> (96,)
SD[4]: stages.0.blocks.0.gamma -> (96,)
SD[5]: stages.0.blocks.0.conv_dw.weight -> (96, 1, 7, 7)
SD[6]: stages.0.blocks.0.conv_dw.bias -> (96,)
SD[7]: stages.0.blocks.0.norm.weight -> (96,)
SD[8]: stages.0.blocks.0.norm.bias -> (96,)
SD[9]: stages.0.blocks.0.mlp.fc1.weight -> (384, 96)
SD[10]: stages.0.blocks.0.mlp.fc1.bias -> (384,)
SD[11]: stages.0.blocks.0.mlp.fc2.weight -> (96, 384)
SD[12]: stages.0.blocks.0.mlp.fc2.bias -> (96,)
SD[13]: stages.0.blocks.1.gamma -> (96,)
SD[14]: stages.0.blocks.1.conv_dw.weight -> (96, 1, 7, 7)
SD[15]: stages.0.blocks.1.conv_dw.bias -> (96,)
SD[16]: stages.0.blocks.1.norm.weight -> (96,)
SD[17]: stages.0.blocks.1.norm.bias -> (96,)
SD[18]: stages.0.blocks.1.mlp.fc1.weight -> (384, 96)
SD[19]